In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
import os
import sys
parent_dir = os.path.dirname(os.environ["GTE_DIR"].replace("Glaciation_time_estimator",""))
GTE_DIR=os.environ["GTE_DIR"]
sys.path.insert(0, parent_dir)
from Glaciation_time_estimator.Auxiliary_func.config_reader import read_config

In [2]:
# config = read_config(
#     os.path.join(GTE_DIR,'configs/2021_tracking/01_01.yaml'))
analyze_year=True
year=2022
global global_rmse
# global_rmse = config["Global_sqrt_mse"]
ice_cont_crit_frac = 0.05
# classifiacation_palette = ['#e41a1c', '#377eb8', "#4daf4a"]
classifiacation_palette = ['#e41a1c', '#377eb8', "#4daf4a"]

In [3]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return 'DJF'
    elif month in [3, 4, 5]:
        return 'MAM'
    elif month in [6, 7, 8]:
        return 'JJA'
    else:
        return 'SON'
if os.uname()[1]=="n2o":
    combined_cloud_df = pd.read_parquet(f"/wolke_scratch/dnikolo/Final_results/{year}_all.parquet")
    glaciations_df = pd.read_parquet(f"/wolke_scratch/dnikolo/Final_results/{year}_glac_04.parquet")
if os.uname()[1][:3]=="eu-":
    combined_cloud_df = pd.read_parquet(f"/cluster/work/climate/dnikolo/Cloud_analysis/full_years/{year}_all.parquet")
    glaciations_df = pd.read_parquet(f"/cluster/work/climate/dnikolo/Cloud_analysis/full_years/{year}_glac_04_thresh.parquet") #_{int(glac_threshold*10):02}_thresh

combined_cloud_df=combined_cloud_df[~combined_cloud_df.is_large_pix_cloud]
combined_cloud_df = combined_cloud_df[(combined_cloud_df.avg_lat >30) | (combined_cloud_df.avg_lat<-30)]

glaciations_df=glaciations_df[~glaciations_df.is_large_pix_cloud]
glaciations_df = glaciations_df[(glaciations_df.avg_lat >30) | (glaciations_df.avg_lat<-30)]


combined_cloud_df['Season'] = combined_cloud_df['track_start_time'].dt.month.apply(month_to_season)
glaciations_df["Radius [km]"]=np.sqrt(glaciations_df["avg_size[km]"]/np.pi)
glaciations_df['Season'] = glaciations_df['track_start_time'].dt.month.apply(month_to_season)
glaciating_clouds = glaciations_df.drop_duplicates(subset="Cloud_ID",keep="first")
combined_cloud_df['is_glaciating'] = combined_cloud_df.index.isin(glaciating_clouds['Cloud_ID'])
combined_cloud_df["month"]= pd.to_datetime(combined_cloud_df['track_start_time']).dt.month
assert glaciating_clouds["Cloud_ID"].isin(combined_cloud_df.index).all(),"The files don't correspont to each other"

In [6]:
era_5_data = xr.load_dataset("/wolke_scratch/dnikolo/ERA5_Data/jan_1_5.nc")

In [61]:
era_5_data

<xarray.Dataset> Size: 445MB
Dimensions:     (valid_time: 120, longitude: 681, latitude: 681)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 960B 2022-01-01 ... 2022-01-05T23...
  * longitude   (longitude) float64 5kB -85.0 -84.75 -84.5 ... 84.5 84.75 85.0
  * latitude    (latitude) float64 5kB 85.0 84.75 84.5 ... -84.5 -84.75 -85.0
Data variables:
    t2m         (valid_time, latitude, longitude) float32 223MB 250.4 ... 243.9
    sst         (valid_time, latitude, longitude) float32 223MB 271.5 ... nan
Attributes:
    CDI:                     Climate Data Interface version 2.3.0 (https://mp...
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    history:                 Mon Apr 28 15:56:54 2025: cdo seltimestep,1/120 ...
    CDO:                     Climate Data Operators version 2.3.0 (https://mp...

In [62]:
t2m = era_5_data["t2m"] 
sst = era_5_data["sst"]
t2m

<xarray.DataArray 't2m' (valid_time: 120, latitude: 681, longitude: 681)> Size: 223MB
array([[[250.44957, 250.45152, 250.45348, ..., 246.0023 , 245.99644,
         245.99059],
        [250.16246, 250.1859 , 250.21129, ..., 246.03941, 246.03355,
         246.02574],
        [249.7484 , 249.74059, 249.73473, ..., 246.02965, 246.01793,
         246.00621],
        ...,
        [257.33435, 257.43787, 257.53943, ..., 242.61168, 242.68199,
         242.7523 ],
        [257.63513, 257.81482, 257.99255, ..., 243.44176, 243.51793,
         243.5941 ],
        [257.49255, 257.69763, 257.9027 , ..., 244.30309, 244.36168,
         244.42223]],

       [[250.41429, 250.40648, 250.40062, ..., 245.91234, 245.90453,
         245.89671],
        [250.1389 , 250.15843, 250.17796, ..., 245.95921, 245.9514 ,
         245.94164],
        [249.73656, 249.72679, 249.71898, ..., 245.96312, 245.95335,
         245.94164],
...
        [255.18738, 255.2362 , 255.28699, ..., 242.78894, 242.80457,
         242.82019],
        [255.44128, 255.55847, 255.67566, ..., 243.3241 , 243.32605,
         243.328  ],
        [255.07605, 255.2284 , 255.38074, ..., 243.72449, 243.70105,
         243.67761]],

       [[246.21635, 246.14018, 246.06596, ..., 244.0562 , 244.05034,
         244.04448],
        [245.79057, 245.75542, 245.7183 , ..., 243.94096, 243.9312 ,
         243.91948],
        [245.29057, 245.23198, 245.17339, ..., 243.89995, 243.89214,
         243.88432],
        ...,
        [254.94292, 254.99565, 255.04839, ..., 243.148  , 243.17729,
         243.20659],
        [255.23393, 255.35112, 255.46635, ..., 243.6187 , 243.64018,
         243.66167],
        [254.89214, 255.04643, 255.20073, ..., 243.89604, 243.88823,
         243.88042]]], dtype=float32)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 960B 2022-01-01 ... 2022-01-05T23...
  * longitude   (longitude) float64 5kB -85.0 -84.75 -84.5 ... 84.5 84.75 85.0
  * latitude    (latitude) float64 5kB 85.0 84.75 84.5 ... -84.5 -84.75 -85.0
Attributes: (12/32)
    standard_name:                            unknown
    long_name:                                2 metre temperature
    units:                                    K
    GRIB_paramId:                             167
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      463761
    ...                                       ...
    GRIB_missingValue:                        3.4028234663852886e+38
    GRIB_name:                                2 metre temperature
    GRIB_shortName:                           2t
    GRIB_totalNumber:                         0
    GRIB_units:                               K
    GRIB_surface:                             0.0

In [29]:
times = t2m.coords["valid_time"].values

In [47]:
clouds_5_days = combined_cloud_df[(combined_cloud_df['track_start_time']+combined_cloud_df['track_length'])<times[-1]]
glaciating_5_days = glaciating_clouds[(glaciating_clouds['track_start_time']+glaciating_clouds['track_length'])<times[-1]]

In [49]:
glaciating_5_days.keys()

Index(['Cloud_ID', 'Time [m]', 'Magnitude', 'Glac_start_ind', 'Glac_peak_ind',
       'Linear', 'line_rmse', 'Rate_arr', 'Mean_glac_rate',
       'Glaciation time [h]', 'is_large_pix_cloud', 'is_cot_valid_cloud',
       'is_ctp_valid_cloud', 'is_liq', 'is_mix', 'is_ice', 'max_water_frac',
       'max_ice_fraction', 'avg_size[km]', 'max_size[km]', 'min_size[km]',
       'avg_size[px]', 'max_size[px]', 'min_size[px]', 'track_start_time',
       'track_length', 'avg_cot', 'avg_ctp', 'glaciation_start_time',
       'glaciation_end_time', 'avg_lat', 'avg_lon', 'start_ice_fraction',
       'end_ice_fraction', 'ice_frac_hist', 'cot_hist', 'cot_nan_frac_hist',
       'ctp_hist', 'ctp_nan_frac_hist', 'lat_hist', 'lon_hist', 'size_hist_km',
       'min_temp', 'max_temp', 'pole', 'Hemisphere', 'Lifetime [h]',
       'Radius [km]', 'Level', 'Optical Thickness', 'Cloud type', 'Season'],
      dtype='object')

In [92]:
sc_5_days = glaciating_5_days[glaciating_5_days["Cloud type"]=="Stratocumulus"]

In [93]:
sc_5_days["mean_temp"]=(sc_5_days["min_temp"]+sc_5_days["max_temp"])/2

/tmp/ipykernel_46157/3875720917.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sc_5_days["mean_temp"]=(sc_5_days["min_temp"]+sc_5_days["max_temp"])/2


In [94]:
sc_5_days["mean_temp"]

0      -9.0
39     -9.0
47     -9.0
59     -9.0
110    -9.0
116    -9.0
121    -9.0
294   -15.0
299   -15.0
495   -15.0
527   -15.0
551   -15.0
571   -15.0
608   -15.0
Name: mean_temp, dtype: float64

In [95]:
def round_quarter(x):
    return round(x * 4) / 4

In [100]:
sc_5_days['temp_diff']

0      [[4.6424972688428e-310, 0.0, 0.0, 0.0, 0.0, 0....
39                                                   NaN
47                                                   NaN
59                                                   NaN
110                                                  NaN
116                                                  NaN
121                                                  NaN
294                                                  NaN
299                                                  NaN
495                                                  NaN
527                                                  NaN
551                                                  NaN
571                                                  NaN
608                                                  NaN
Name: temp_diff, dtype: object

In [109]:
# before the loop
# sc_5_days['temp_diff'] = pd.Series(, dtype='object')

sc_5_days['is_sea']    = True          # defaults, will flip to False if needed
for cloud_ind,cloud in sc_5_days.iterrows():
    # sc_5_days.loc[cloud_ind, 'temp_diff'] = np.empty(len(cloud.lat_hist))
    # sc_5_days.loc[cloud_ind,"is_sea"] = True
    t_diff = np.empty(len(cloud.lat_hist))
    for loc_ind,lat in enumerate(cloud.lat_hist):
        lon=cloud.lon_hist[loc_ind]
        time = cloud.track_start_time + pd.Timedelta(minutes=15)*loc_ind
        sst_val = sst.sel(valid_time=time.round('h'), latitude=round_quarter(lat), longitude=round_quarter(lon)).values
        if np.isnan(sst_val):
            sc_5_days.loc[cloud_ind,"is_sea"] = False
            break
        #     continue
        # TODO: Make it use t2m instaed of sst
        t2m_val = t2m.sel(valid_time=time.round('h'), latitude=round_quarter(lat), longitude=round_quarter(lon)).values
        t_diff[loc_ind] = t2m_val - (cloud["mean_temp"]+273.15)
    print(t_diff.mean(),t_diff.std(),t_diff.min(),t_diff.max())    
    sc_5_days.loc[cloud_ind,"t_diff"] = t_diff.mean()
    

/tmp/ipykernel_46157/20028790.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sc_5_days['is_sea']    = True          # defaults, will flip to False if needed


21.329446411132835 0.11308608322760065 21.104425048828148 21.540948486328148
17.05827849014948 0.6941163791999319 15.493890380859398 17.807397460937523
13.34144576767746 1.157308997240438 11.059655761718773 15.560083007812523
19.4783394949777 0.37551754863002523 18.925561523437523 19.882592773437523
10.602807617187523 0.13028270187111846 10.376367187500023 10.741906738281273
19.517686462402366 0.15143676493089597 19.272729492187523 19.790307617187523
18.472520446777366 0.39690748308181373 17.818627929687523 18.972924804687523
19.89112243652346 0.2213170481044886 19.600427246093773 20.350671386718773
14.294824218750023 0.28295845775538786 13.995507812500023 14.577539062500023
14.622743918679 0.021142441854688873 14.584771728515648 14.657220458984398
15.347924804687523 0.01686559414990643 15.326684570312523 15.367883300781273
15.382280706590244 0.2240202493708793 14.910791015625023 15.727868652343773
14.031349012586828 0.058084028245689326 13.983056640625023 14.164941406250023
15.3066548

In [83]:
a = sst.sel(valid_time=times[0],latitude=26,longitude=0).values
if np.isnan(a):
    print("nan")

nan


In [106]:
sc_5_days['t_diff']*sc_5_days["avg_cth"]

KeyError: 'avg_cth'

In [107]:
times[1]

numpy.datetime64('2022-01-01T01:00:00.000000000')

In [ ]:
t2m.interp(valid_time=times[0], latitude=75.3, longitude=0).values


array(262.68083496)